In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Основная папка проекта
PROJECT_DIR = Path(r"D:\RLMS_VKR")

# Папки проекта
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
TABLES_DIR = RESULTS_DIR / "tables"

# Создаём папку для таблиц, если её ещё нет
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Рабочая папка:", PROJECT_DIR)
print("Папка с таблицами:", TABLES_DIR)

Рабочая папка: D:\RLMS_VKR
Папка с таблицами: D:\RLMS_VKR\results\tables


In [2]:
def classify_ilo(j77, j1, j81, j82, j84_1, j84_2):
    """
    Приближённая классификация по критериям МОТ.

    E  - занят
    U  - безработный
    I  - вне рабочей силы
    NA - недостаточно данных

    Важно: это исследовательская классификация
    по доступным переменным RLMS-HSE.
    """

    def missing(x):
        return pd.isna(x)

    result = {
        "status_ilo": "NA",
        "ilo_subgroup": None,
        "classification_quality": "low",
        "classification_reason": "Недостаточно информации"
    }

    # 1. Определяем наличие работы

    if not missing(j77) and j77 == 1:
        result.update({
            "status_ilo": "E",
            "ilo_subgroup": None,
            "classification_quality": "high",
            "classification_reason": "По j77 подтверждено наличие работы"
        })
        return result

    if not missing(j77) and j77 == 2:
        has_job = False
        employment_quality = "high"

    elif missing(j77) and not missing(j1):
        if j1 == 1:
            result.update({
                "status_ilo": "E",
                "classification_quality": "medium",
                "classification_reason": "j77 отсутствует, j1 указывает на работу"
            })
            return result

        elif j1 == 5:
            has_job = False
            employment_quality = "medium"

        else:
            result["classification_reason"] = (
                "По j1 недостаточно информации о наличии работы"
            )
            return result

    else:
        result["classification_reason"] = (
            "Не удалось определить наличие или отсутствие работы"
        )
        return result

    # 2. Проверяем особый случай: работа уже найдена

    if not missing(j84_2) and j84_2 == 99999995:
        result.update({
            "status_ilo": "NA",
            "ilo_subgroup": "future_job",
            "classification_quality": "medium",
            "classification_reason": (
                "Работа уже найдена, но срок начала и доступность "
                "требуют дополнительной проверки"
            )
        })
        return result

    # 3. Проверяем активный поиск работы

    if not missing(j82) and j82 == 1:

        if not missing(j84_1) and j84_1 == 1:
            result.update({
                "status_ilo": "U",
                "ilo_subgroup": None,
                "classification_quality": "medium",
                "classification_reason": (
                    "Нет работы, поиск за 30 дней, "
                    "готовность приступить немедленно"
                )
            })
            return result

        if not missing(j84_1) and j84_1 == 2:
            result.update({
                "status_ilo": "NA",
                "ilo_subgroup": "availability_unknown",
                "classification_quality": "low",
                "classification_reason": (
                    "Искал работу, но возможность приступить "
                    "в течение двух недель не установлена"
                )
            })
            return result

        result["classification_reason"] = (
            "Поиск работы подтверждён, но готовность неизвестна"
        )
        return result

    # 4. Проверяем отсутствие поиска

    if not missing(j82) and j82 == 2:

        if not missing(j81) and j81 == 1:
            result.update({
                "status_ilo": "I",
                "ilo_subgroup": "wants_job",
                "classification_quality": "medium",
                "classification_reason": (
                    "Не искал работу, но выразил желание работать; "
                    "относится к населению вне рабочей силы"
                )
            })
            return result

        if not missing(j81) and j81 == 2:
            result.update({
                "status_ilo": "I",
                "ilo_subgroup": None,
                "classification_quality": "medium",
                "classification_reason": (
                    "Не искал работу и не выразил желания её найти"
                )
            })
            return result

        result["classification_reason"] = (
            "Не искал работу, но желание работать неизвестно"
        )
        return result

    result["classification_reason"] = (
        "Недостаточно данных о поиске работы"
    )

    return result

In [3]:
def classify_abm(j77, j1, j81, j82, j136, j84_1, j84_2):
    """
    Расширенная операциональная классификация
    для агентной модели.

    E  - занятый
    U  - активный безработный
    P1 - хочет работать, не ищет, готов приступить
    P2 - ищет работу, но не готов приступить немедленно
    P3 - хочет работать, не ищет и не готов приступить
    I  - экономически неактивный
    F  - ожидает начала уже найденной работы
    NA - недостаточно данных
    """

    def missing(x):
        return pd.isna(x)

    result = {
        "status_abm": "NA",
        "classification_quality": "low",
        "classification_reason": "Недостаточно информации"
    }

    # 1. Определяем занятость

    if not missing(j77) and j77 == 1:
        result.update({
            "status_abm": "E",
            "classification_quality": "high",
            "classification_reason": "По j77 подтверждено наличие работы"
        })
        return result

    if not missing(j77) and j77 == 2:
        no_job = True

    elif missing(j77) and not missing(j1):
        if j1 == 1:
            result.update({
                "status_abm": "E",
                "classification_quality": "medium",
                "classification_reason": "j77 отсутствует, j1 указывает на работу"
            })
            return result

        elif j1 == 5:
            no_job = True

        else:
            result["classification_reason"] = (
                "Необходимо уточнить наличие работы по j1"
            )
            return result

    else:
        result["classification_reason"] = (
            "Невозможно определить наличие работы"
        )
        return result

    # 2. Уже найденная работа

    if not missing(j84_2) and j84_2 == 99999995:
        result.update({
            "status_abm": "F",
            "classification_quality": "medium",
            "classification_reason": (
                "Работа найдена, респондент ожидает начала работы"
            )
        })
        return result

    # 3. Человек ищет работу

    if not missing(j82) and j82 == 1:

        if not missing(j84_1) and j84_1 == 1:
            result.update({
                "status_abm": "U",
                "classification_quality": "medium",
                "classification_reason": (
                    "Ищет работу и готов приступить немедленно"
                )
            })
            return result

        if not missing(j84_1) and j84_1 == 2:
            result.update({
                "status_abm": "P2",
                "classification_quality": "medium",
                "classification_reason": (
                    "Ищет работу, но не может приступить немедленно"
                )
            })
            return result

        result["classification_reason"] = (
            "Ищет работу, но готовность неизвестна"
        )
        return result

    # 4. Человек не ищет работу

    if not missing(j82) and j82 == 2:

        if not missing(j81) and j81 == 1:

            if not missing(j84_1) and j84_1 == 1:
                result.update({
                    "status_abm": "P1",
                    "classification_quality": "medium",
                    "classification_reason": (
                        "Хочет работать, не ищет работу, "
                        "готов приступить немедленно"
                    )
                })
                return result

            if not missing(j84_1) and j84_1 == 2:
                result.update({
                    "status_abm": "P3",
                    "classification_quality": "medium",
                    "classification_reason": (
                        "Хочет работать, не ищет работу "
                        "и не может приступить немедленно"
                    )
                })
                return result

            result["classification_reason"] = (
                "Хочет работать, но готовность неизвестна"
            )
            return result

        if not missing(j81) and j81 == 2:
            result.update({
                "status_abm": "I",
                "classification_quality": "medium",
                "classification_reason": (
                    "Не ищет работу и не хочет её найти"
                )
            })
            return result

        result["classification_reason"] = (
            "Не ищет работу, но желание работать неизвестно"
        )
        return result

    result["classification_reason"] = (
        "Недостаточно данных о поиске работы"
    )

    return result

In [4]:
# Тестовые респонденты
test_data = pd.DataFrame([
    # j77, j1, j81, j82, j136, j84_1, j84_2

    [1, 1, 1, 1, 1, 1, 10],          # Есть работа
    [2, 5, 1, 1, 1, 1, 15],          # Без работы, ищет, готов
    [2, 5, 1, 1, 1, 2, 20],          # Ищет, не готов сразу
    [2, 5, 1, 2, np.nan, 1, 30],     # Хочет, не ищет, готов
    [2, 5, 1, 2, np.nan, 2, 30],     # Хочет, не ищет, не готов
    [2, 5, 2, 2, np.nan, 2, 99999996], # Не хочет работать
    [2, 5, 1, 1, 1, np.nan, 20],     # Неизвестна готовность
    [2, 5, 1, 1, 1, 1, 99999995],    # Уже нашёл работу
    [np.nan, 1, 1, 1, 1, 1, 10],     # Волна без j77, работает
    [np.nan, 5, 1, 1, 1, 1, 10]      # Волна без j77, нет работы
], columns=[
    "j77", "j1", "j81", "j82",
    "j136", "j84_1", "j84_2"
])

# Применяем обе классификации
ilo_results = test_data.apply(
    lambda row: classify_ilo(
        row["j77"], row["j1"], row["j81"],
        row["j82"], row["j84_1"], row["j84_2"]
    ),
    axis=1
)

abm_results = test_data.apply(
    lambda row: classify_abm(
        row["j77"], row["j1"], row["j81"],
        row["j82"], row["j136"],
        row["j84_1"], row["j84_2"]
    ),
    axis=1
)

# Формируем итоговую таблицу
test_results = pd.concat([
    test_data,
    pd.DataFrame(ilo_results.tolist()),
    pd.DataFrame(abm_results.tolist()).add_prefix("abm_")
], axis=1)

display(test_results)

,j77,j1,j81,j82,j136,j84_1,j84_2,status_ilo,ilo_subgroup,classification_quality,classification_reason,abm_status_abm,abm_classification_quality,abm_classification_reason
0,1.0,1,1,1,1.0,1.0,10,E,NaN,high,По j77 подтверждено наличие работы,E,high,По j77 подтверждено наличие работы
1,2.0,5,1,1,1.0,1.0,15,U,NaN,medium,"Нет работы, поиск за 30 дней, готовность прист...",U,medium,Ищет работу и готов приступить немедленно
2,2.0,5,1,1,1.0,2.0,20,NA,availability_unknown,low,"Искал работу, но возможность приступить в тече...",P2,medium,"Ищет работу, но не может приступить немедленно"
3,2.0,5,1,2,NaN,1.0,30,I,wants_job,medium,"Не искал работу, но выразил желание работать; ...",P1,medium,"Хочет работать, не ищет работу, готов приступи..."
4,2.0,5,1,2,NaN,2.0,30,I,wants_job,medium,"Не искал работу, но выразил желание работать; ...",P3,medium,"Хочет работать, не ищет работу и не может прис..."
5,2.0,5,2,2,NaN,2.0,99999996,I,NaN,medium,Не искал работу и не выразил желания её найти,I,medium,Не ищет работу и не хочет её найти
6,2.0,5,1,1,1.0,NaN,20,NA,NaN,low,"Поиск работы подтверждён, но готовность неизве...",NA,low,"Ищет работу, но готовность неизвестна"
7,2.0,5,1,1,1.0,1.0,99999995,NA,future_job,medium,"Работа уже найдена, но срок начала и доступнос...",F,medium,"Работа найдена, респондент ожидает начала работы"
8,NaN,1,1,1,1.0,1.0,10,E,NaN,medium,"j77 отсутствует, j1 указывает на работу",E,medium,"j77 отсутствует, j1 указывает на работу"
9,NaN,5,1,1,1.0,1.0,10,U,NaN,medium,"Нет работы, поиск за 30 дней, готовность прист...",U,medium,Ищет работу и готов приступить немедленно


In [5]:
def is_missing(value):
    """Проверка пропущенного значения."""
    return pd.isna(value)


def get_employment_status(j77, j1):
    """
    Определение наличия работы.

    Возвращает:
    employed, quality, reason
    """

    if not is_missing(j77):

        if j77 == 1:
            return True, "high", "j77 = 1: есть работа"

        if j77 == 2:
            if not is_missing(j1) and j1 == 1:
                return None, "low", (
                    "Противоречие: j77 = 2, j1 = 1"
                )

            return False, "high", "j77 = 2: работы нет"

        return None, "low", "Специальный код j77"

    # Если j77 отсутствует, используем j1
    if not is_missing(j1):

        if j1 == 1:
            return True, "medium", (
                "j77 отсутствует, j1 = 1: указана работа"
            )

        if j1 == 5:
            return False, "medium", (
                "j77 отсутствует, j1 = 5: указано отсутствие работы"
            )

        return None, "low", (
            "По j1 нельзя однозначно определить занятость"
        )

    return None, "low", "Нет данных о занятости"


def get_search_information(j82, j83_values, j136):
    """
    Сохраняет информацию о поиске работы.

    Не приравнивает наличие способа поиска
    к подтверждённому критерию МОТ.
    """

    j83_values = list(j83_values)

    j83_yes = sum(
        1 for value in j83_values
        if not is_missing(value) and value == 1
    )

    j83_no = sum(
        1 for value in j83_values
        if not is_missing(value) and value == 2
    )

    if not is_missing(j82) and j82 == 1:
        search_reported = True
    elif not is_missing(j82) and j82 == 2:
        search_reported = False
    else:
        search_reported = None

    if not is_missing(j136) and j136 == 1:
        search_7d = True
    elif not is_missing(j136) and j136 == 2:
        search_7d = False
    else:
        search_7d = None

    return {
        "search_reported": search_reported,
        "search_7d": search_7d,
        "j83_yes_count": j83_yes,
        "j83_no_count": j83_no,
        "j83_any_yes": j83_yes > 0,
        "active_search_verified": None
    }

In [9]:
def classify_ilo(
    j77, j1, j81, j82, j136, j84_1, j84_2,
    j83_values,
    active_search_verified=None
):
    """
    Приближённая классификация по критериям МОТ.

    E  - занятый
    U  - безработный
    I  - вне рабочей силы
    NA - недостаточно информации

    Дополнительно:
    ilo_subgroup - уточняющая подгруппа.

    Важно:
    Классификация основана на доступных переменных RLMS-HSE
    и не является полной официальной классификацией МОТ.
    """

    result = {
        "status_ilo": "NA",
        "ilo_subgroup": None,
        "classification_quality": "low",
        "classification_reason": "Недостаточно информации"
    }

    # 1. Проверяем наличие работы

    employed, quality, reason = get_employment_status(j77, j1)

    if employed is True:
        result.update({
            "status_ilo": "E",
            "ilo_subgroup": None,
            "classification_quality": quality,
            "classification_reason": reason
        })
        return result

    if employed is None:
        result["classification_reason"] = reason
        return result

    # 2. Проверяем, не нашёл ли человек уже работу

    if not is_missing(j84_2) and j84_2 == 99999995:
        result.update({
            "status_ilo": "NA",
            "ilo_subgroup": "future_job",
            "classification_quality": "low",
            "classification_reason": (
                "Работа уже найдена, но дата начала "
                "и доступность не подтверждены"
            )
        })
        return result

    # 3. Получаем информацию о поиске работы

    search = get_search_information(
        j82,
        j83_values,
        j136
    )

    # 4. Человек искал работу

    if search["search_reported"] is True:

        # Активный поиск должен быть отдельно подтверждён
        if active_search_verified is not True:
            result.update({
                "ilo_subgroup": "search_not_verified",
                "classification_quality": "low",
                "classification_reason": (
                    "Поиск за последние 30 дней заявлен, "
                    "но активные действия по критериям МОТ "
                    "не подтверждены"
                )
            })
            return result

        # Готовность приступить немедленно подтверждена
        if not is_missing(j84_1) and j84_1 == 1:
            result.update({
                "status_ilo": "U",
                "ilo_subgroup": None,
                "classification_quality": "medium",
                "classification_reason": (
                    "Нет работы, активный поиск подтверждён, "
                    "готов приступить немедленно"
                )
            })
            return result

        # Ответ 2 не доказывает недоступность в течение 2 недель
        if not is_missing(j84_1) and j84_1 == 2:
            result.update({
                "ilo_subgroup": "availability_unknown",
                "classification_quality": "low",
                "classification_reason": (
                    "Искал работу, но возможность приступить "
                    "в течение двух недель не установлена"
                )
            })
            return result

        result["classification_reason"] = (
            "Активный поиск подтверждён, "
            "но готовность приступить неизвестна"
        )
        return result

    # 5. Человек не искал работу

    if search["search_reported"] is False:

        # Человек хочет работать
        if not is_missing(j81) and j81 == 1:

            if not is_missing(j84_1) and j84_1 == 1:
                subgroup = "potential_labour_force_candidate"
                reason = (
                    "Не искал работу, хочет работать "
                    "и готов приступить немедленно"
                )
            else:
                subgroup = "wants_job_availability_unknown"
                reason = (
                    "Не искал работу, хочет работать, "
                    "но доступность в течение двух недель "
                    "не установлена"
                )

            result.update({
                "status_ilo": "I",
                "ilo_subgroup": subgroup,
                "classification_quality": "medium",
                "classification_reason": reason
            })
            return result

        # Человек не хочет работать
        if not is_missing(j81) and j81 == 2:
            result.update({
                "status_ilo": "I",
                "ilo_subgroup": None,
                "classification_quality": "medium",
                "classification_reason": (
                    "Не искал работу и не выразил желания её найти"
                )
            })
            return result

    # 6. Если информации недостаточно

    result["classification_reason"] = (
        "Недостаточно данных для определения статуса"
    )

    return result

In [10]:
def classify_abm(
    j77, j1, j81, j82, j136, j84_1, j84_2,
    j83_values
):
    """
    Расширенная классификация для агентной модели.

    E  - занятый
    U  - активный безработный
    P1 - хочет работать, не ищет, готов приступить
    P2 - ищет работу, но не готов приступить немедленно
    P3 - хочет работать, не ищет, не готов приступить немедленно
    I  - экономически неактивный
    F  - ожидает начала уже найденной работы
    NA - недостаточно информации
    """

    result = {
        "status_abm": "NA",
        "classification_quality": "low",
        "classification_reason": "Недостаточно информации"
    }

    # 1. Проверяем занятость
    employed, quality, reason = get_employment_status(j77, j1)

    if employed is True:
        result.update({
            "status_abm": "E",
            "classification_quality": quality,
            "classification_reason": reason
        })
        return result

    if employed is None:
        result["classification_reason"] = reason
        return result

    # 2. Проверяем найденную работу
    if not is_missing(j84_2) and j84_2 == 99999995:
        result.update({
            "status_abm": "F",
            "classification_quality": "medium",
            "classification_reason": (
                "Работа уже найдена, ожидает начала работы"
            )
        })
        return result

    # 3. Сохраняем информацию о поиске
    search = get_search_information(j82, j83_values, j136)

    # 4. Человек искал работу
    if search["search_reported"] is True:

        if not is_missing(j84_1) and j84_1 == 1:
            result.update({
                "status_abm": "U",
                "classification_quality": "medium",
                "classification_reason": (
                    "Нет работы, ищет работу и готов приступить немедленно"
                )
            })
            return result

        if not is_missing(j84_1) and j84_1 == 2:
            result.update({
                "status_abm": "P2",
                "classification_quality": "medium",
                "classification_reason": (
                    "Ищет работу, но не может приступить немедленно; "
                    "доступность в течение двух недель неизвестна"
                )
            })
            return result

        result["classification_reason"] = (
            "Ищет работу, но готовность приступить неизвестна"
        )
        return result

    # 5. Человек не искал работу
    if search["search_reported"] is False:

        if not is_missing(j81) and j81 == 1:

            if not is_missing(j84_1) and j84_1 == 1:
                result.update({
                    "status_abm": "P1",
                    "classification_quality": "medium",
                    "classification_reason": (
                        "Хочет работать, не искал работу, "
                        "готов приступить немедленно"
                    )
                })
                return result

            if not is_missing(j84_1) and j84_1 == 2:
                result.update({
                    "status_abm": "P3",
                    "classification_quality": "medium",
                    "classification_reason": (
                        "Хочет работать, не искал работу, "
                        "не готов приступить немедленно"
                    )
                })
                return result

            result["classification_reason"] = (
                "Хочет работать, но готовность неизвестна"
            )
            return result

        if not is_missing(j81) and j81 == 2:
            result.update({
                "status_abm": "I",
                "classification_quality": "medium",
                "classification_reason": (
                    "Не ищет работу и не хочет её найти"
                )
            })
            return result

    result["classification_reason"] = (
        "Недостаточно данных для расширенной классификации"
    )

    return result

In [11]:
test_data = pd.DataFrame([
    [1, 1, 1, 1, 1, 1, 10, [1, 2, 2, 2, 2, 2, np.nan, np.nan], True],
    [2, 5, 1, 1, 1, 1, 15, [1, 2, 2, 2, 2, 2, np.nan, np.nan], True],
    [2, 5, 1, 1, 1, 2, 20, [1, 2, 2, 2, 2, 2, np.nan, np.nan], True],
    [2, 5, 1, 2, np.nan, 1, 30, [np.nan]*8, None],
    [2, 5, 1, 2, np.nan, 2, 30, [np.nan]*8, None],
    [2, 5, 2, 2, np.nan, 2, 99999996, [np.nan]*8, None],
    [2, 5, 1, 1, 1, np.nan, 20, [1, 2, 2, 2, 2, 2, np.nan, np.nan], True],
    [2, 5, 1, 1, 1, 1, 99999995, [1, 2, 2, 2, 2, 2, np.nan, np.nan], True],
    [np.nan, 1, 1, 1, 1, 1, 10, [1, 2, 2, 2, 2, 2, np.nan, np.nan], True],
    [np.nan, 5, 1, 1, 1, 1, 10, [1, 2, 2, 2, 2, 2, np.nan, np.nan], True]
], columns=[
    "j77", "j1", "j81", "j82", "j136",
    "j84_1", "j84_2", "j83_values",
    "active_search_verified"
])

ilo_results = test_data.apply(
    lambda row: classify_ilo(
        row["j77"], row["j1"], row["j81"],
        row["j82"], row["j136"], row["j84_1"],
        row["j84_2"], row["j83_values"],
        row["active_search_verified"]
    ),
    axis=1
)

abm_results = test_data.apply(
    lambda row: classify_abm(
        row["j77"], row["j1"], row["j81"],
        row["j82"], row["j136"], row["j84_1"],
        row["j84_2"], row["j83_values"]
    ),
    axis=1
)

test_results = pd.concat([
    test_data.drop(columns=["j83_values", "active_search_verified"]),
    pd.DataFrame(ilo_results.tolist()),
    pd.DataFrame(abm_results.tolist())
], axis=1)

display(test_results)

,j77,j1,j81,j82,j136,j84_1,j84_2,status_ilo,ilo_subgroup,classification_quality,classification_reason,status_abm,classification_quality,classification_reason
0,1.0,1,1,1,1.0,1.0,10,E,NaN,high,j77 = 1: есть работа,E,high,j77 = 1: есть работа
1,2.0,5,1,1,1.0,1.0,15,U,NaN,medium,"Нет работы, активный поиск подтверждён, готов ...",U,medium,"Нет работы, ищет работу и готов приступить нем..."
2,2.0,5,1,1,1.0,2.0,20,NA,availability_unknown,low,"Искал работу, но возможность приступить в тече...",P2,medium,"Ищет работу, но не может приступить немедленно..."
3,2.0,5,1,2,NaN,1.0,30,I,potential_labour_force_candidate,medium,"Не искал работу, хочет работать и готов присту...",P1,medium,"Хочет работать, не искал работу, готов приступ..."
4,2.0,5,1,2,NaN,2.0,30,I,wants_job_availability_unknown,medium,"Не искал работу, хочет работать, но доступност...",P3,medium,"Хочет работать, не искал работу, не готов прис..."
5,2.0,5,2,2,NaN,2.0,99999996,I,NaN,medium,Не искал работу и не выразил желания её найти,I,medium,Не ищет работу и не хочет её найти
6,2.0,5,1,1,1.0,NaN,20,NA,NaN,low,"Активный поиск подтверждён, но готовность прис...",NA,low,"Ищет работу, но готовность приступить неизвестна"
7,2.0,5,1,1,1.0,1.0,99999995,NA,future_job,low,"Работа уже найдена, но дата начала и доступнос...",F,medium,"Работа уже найдена, ожидает начала работы"
8,NaN,1,1,1,1.0,1.0,10,E,NaN,medium,"j77 отсутствует, j1 = 1: указана работа",E,medium,"j77 отсутствует, j1 = 1: указана работа"
9,NaN,5,1,1,1.0,1.0,10,U,NaN,medium,"Нет работы, активный поиск подтверждён, готов ...",U,medium,"Нет работы, ищет работу и готов приступить нем..."


In [13]:
# Загрузка метаданных RLMS-HSE

import pyreadstat

DATA_PATH = DATA_DIR / "RLMS_IND_1994_2025_rus.dta"

_, meta = pyreadstat.read_dta(
    str(DATA_PATH),
    metadataonly=True,
    encoding="utf-8"
)

print("Метаданные успешно загружены.")
print("Количество наблюдений:", meta.number_rows)
print("Количество переменных:", meta.number_columns)

Метаданные успешно загружены.
Количество наблюдений: 474600
Количество переменных: 4283


In [14]:
# Формулировки вопросов о способах поиска работы

j83_variables = [
    f"j83_{i}" for i in range(1, 9)
]

for var in j83_variables:
    label = meta.column_names_to_labels.get(
        var,
        "Описание отсутствует"
    )

    print(f"{var}: {label}")

j83_1: Вы обращались в поисках работы в государств
j83_2: Вы обращались в поисках работы в негосударс
j83_3: Вы обращались в поисках работы к друзьям, зн
j83_4: Вы обращались в поисках работы к родственни
j83_5: Вы обращались в поисках работы непосредств
j83_6: Вы обращались в поисках работы к рекламным 
j83_7: Вы обращались в поисках работы к рекламным 
j83_8: Вы обращались в поисках работы в социальные


In [15]:
for var in j83_variables:
    label = meta.column_names_to_labels.get(var, "Описание отсутствует")
    print(f"\n{var} ({len(label)} символов):")
    print(repr(label))


j83_1 (43 символов):
'Вы обращались в поисках работы в государств'

j83_2 (43 символов):
'Вы обращались в поисках работы в негосударс'

j83_3 (44 символов):
'Вы обращались в поисках работы к друзьям, зн'

j83_4 (43 символов):
'Вы обращались в поисках работы к родственни'

j83_5 (42 символов):
'Вы обращались в поисках работы непосредств'

j83_6 (43 символов):
'Вы обращались в поисках работы к рекламным '

j83_7 (43 символов):
'Вы обращались в поисках работы к рекламным '

j83_8 (43 символов):
'Вы обращались в поисках работы в социальные'


In [16]:
# Справочник способов поиска работы j83

j83_labels = {
    "j83_1": "Обращение в государственные службы занятости, на биржи труда",
    "j83_2": "Обращение в негосударственные службы занятости",
    "j83_3": "Обращение к друзьям и знакомым",
    "j83_4": "Обращение к родственникам",
    "j83_5": "Непосредственное обращение на предприятия",
    "j83_6": "Обращение к рекламным объявлениям вне Интернета",
    "j83_7": "Обращение к рекламным объявлениям в Интернете",
    "j83_8": "Обращение к социальным сетям"
}

j83_groups = {
    "j83_1": "Организации и работодатели",
    "j83_2": "Организации и работодатели",
    "j83_3": "Личные контакты",
    "j83_4": "Личные контакты",
    "j83_5": "Организации и работодатели",
    "j83_6": "Объявления и интернет",
    "j83_7": "Объявления и интернет",
    "j83_8": "Объявления и интернет"
}

j83_reference = pd.DataFrame({
    "Переменная": list(j83_labels.keys()),
    "Полное название": list(j83_labels.values()),
    "Группа способов поиска": [
        j83_groups[var] for var in j83_labels
    ],
    "Коды ответов": ["1 — да; 2 — нет; специальные коды"] * 8
})

display(j83_reference)

,Переменная,Полное название,Группа способов поиска,Коды ответов
0,j83_1,"Обращение в государственные службы занятости, ...",Организации и работодатели,1 — да; 2 — нет; специальные коды
1,j83_2,Обращение в негосударственные службы занятости,Организации и работодатели,1 — да; 2 — нет; специальные коды
2,j83_3,Обращение к друзьям и знакомым,Личные контакты,1 — да; 2 — нет; специальные коды
3,j83_4,Обращение к родственникам,Личные контакты,1 — да; 2 — нет; специальные коды
4,j83_5,Непосредственное обращение на предприятия,Организации и работодатели,1 — да; 2 — нет; специальные коды
5,j83_6,Обращение к рекламным объявлениям вне Интернета,Объявления и интернет,1 — да; 2 — нет; специальные коды
6,j83_7,Обращение к рекламным объявлениям в Интернете,Объявления и интернет,1 — да; 2 — нет; специальные коды
7,j83_8,Обращение к социальным сетям,Объявления и интернет,1 — да; 2 — нет; специальные коды


In [17]:
j83_reference.to_csv(
    TABLES_DIR / "j83_variables_reference.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Справочник j83 сохранён.")

Справочник j83 сохранён.


In [18]:
def get_search_methods(j83_values):
    """
    Определяет отмеченные респондентом способы поиска работы.
    1 — способ использовался.
    2 — способ не использовался.
    """

    methods = []

    for var, value in zip(j83_labels.keys(), j83_values):
        if value == 1:
            methods.append({
                "variable": var,
                "method": j83_labels[var],
                "group": j83_groups[var]
            })

    return methods

In [19]:
example_j83 = [1, 2, 1, 2, 1, 2, 1, 2]

example_methods = get_search_methods(example_j83)

for method in example_methods:
    print(method["variable"], "-", method["method"])

j83_1 - Обращение в государственные службы занятости, на биржи труда
j83_3 - Обращение к друзьям и знакомым
j83_5 - Непосредственное обращение на предприятия
j83_7 - Обращение к рекламным объявлениям в Интернете


In [20]:
# Разделение способов поиска по степени подтверждения активного поиска

J83_ACTIVE = ["j83_1", "j83_2", "j83_3", "j83_4", "j83_5"]

J83_AMBIGUOUS = ["j83_6", "j83_7", "j83_8"]


def assess_active_search(j83_values):
    """
    Оценивает подтверждение активного поиска работы.

    True  — есть хотя бы один однозначный активный способ.
    None  — отмечены только неоднозначные способы
            либо недостаточно информации.
    False — все способы имеют ответ 'нет'.
    """

    values = dict(zip(j83_labels.keys(), j83_values))

    # Есть хотя бы один подтверждённый активный способ
    for var in J83_ACTIVE:
        if values.get(var) == 1:
            return True

    # Проверяем неоднозначные способы
    for var in J83_AMBIGUOUS:
        if values.get(var) == 1:
            return None

    # Проверяем, есть ли достаточно информации,
    # чтобы утверждать, что активных способов не было
    valid_answers = [1, 2]

    if all(values.get(var) in valid_answers for var in j83_labels):
        return False

    return None

In [21]:
test_cases = {
    "Есть активный способ": [1, 2, 2, 2, 2, 2, 2, 2],
    "Только неоднозначный способ": [2, 2, 2, 2, 2, 1, 2, 2],
    "Все способы отсутствуют": [2, 2, 2, 2, 2, 2, 2, 2],
    "Нет достаточной информации": [2, 2, 2, 2, 2, 99999999, 2, 2]
}

for name, values in test_cases.items():
    print(name, "→", assess_active_search(values))

Есть активный способ → True
Только неоднозначный способ → None
Все способы отсутствуют → False
Нет достаточной информации → None


In [24]:
def classify_ilo(
    j77, j1, j81, j82, j136,
    j84_1, j84_2, j83_values
):
    """
    Классификация респондента по категориям E/U/I.

    E  — занятый;
    U  — безработный;
    I  — вне рабочей силы;
    NA — недостаточно информации для классификации.
    """

    # 1. Определяем наличие работы
    employment_result = get_employment_status(j77, j1)

    employed = employment_result[0]
    confidence = employment_result[1]
    if employed is True:
        return "E", "employed"

    if employed is None:
        return "NA", "employment_unknown"

    # 2. Учитываем ситуацию, когда работа уже найдена
    if j84_2 == 99999995:
        return "NA", "future_job"

    # 3. Проверяем активный поиск работы
    if j82 == 1:

        active_search_verified = assess_active_search(j83_values)

        # Нет подтверждения активного поиска
        if active_search_verified is not True:
            return "NA", "search_not_verified"

        # 4. Проверяем готовность приступить к работе
        if j84_1 == 1:
            return "U", "unemployed"

        if j84_1 == 2:
            return "NA", "availability_unknown"

        return "NA", "availability_missing"

    # 5. Если человек не искал работу
    if j82 == 2:

        # Хотел бы работать
        if j81 == 1:

            if j84_1 == 1:
                return "I", "potential_labour_force_candidate"

            return "I", "wants_job_availability_unknown"

        # Не хотел бы работать
        if j81 == 2:
            return "I", "outside_labour_force"

    # 6. Во всех остальных случаях информации недостаточно
    return "NA", "insufficient_information"

In [25]:
# Проверка интеграции classify_ilo и assess_active_search

test_cases = [
    {
        "name": "Активный поиск подтверждён",
        "j77": 2, "j1": 5, "j81": 1, "j82": 1,
        "j136": 1, "j84_1": 1, "j84_2": np.nan,
        "j83": [1, 2, 2, 2, 2, 2, 2, 2]
    },
    {
        "name": "Только неоднозначный способ поиска",
        "j77": 2, "j1": 5, "j81": 1, "j82": 1,
        "j136": 1, "j84_1": 1, "j84_2": np.nan,
        "j83": [2, 2, 2, 2, 2, 1, 2, 2]
    },
    {
        "name": "Все способы поиска отсутствуют",
        "j77": 2, "j1": 5, "j81": 1, "j82": 1,
        "j136": 1, "j84_1": 1, "j84_2": np.nan,
        "j83": [2, 2, 2, 2, 2, 2, 2, 2]
    },
    {
        "name": "Не искал, но хочет работать и готов приступить",
        "j77": 2, "j1": 5, "j81": 1, "j82": 2,
        "j136": np.nan, "j84_1": 1, "j84_2": np.nan,
        "j83": [np.nan] * 8
    },
    {
        "name": "Не искал и не хочет работать",
        "j77": 2, "j1": 5, "j81": 2, "j82": 2,
        "j136": np.nan, "j84_1": np.nan, "j84_2": np.nan,
        "j83": [np.nan] * 8
    }
]

for case in test_cases:

    result = classify_ilo(
        j77=case["j77"],
        j1=case["j1"],
        j81=case["j81"],
        j82=case["j82"],
        j136=case["j136"],
        j84_1=case["j84_1"],
        j84_2=case["j84_2"],
        j83_values=case["j83"]
    )

    print(case["name"], "→", result)

Активный поиск подтверждён → ('U', 'unemployed')
Только неоднозначный способ поиска → ('NA', 'search_not_verified')
Все способы поиска отсутствуют → ('NA', 'search_not_verified')
Не искал, но хочет работать и готов приступить → ('I', 'potential_labour_force_candidate')
Не искал и не хочет работать → ('I', 'outside_labour_force')


In [26]:
import pyreadstat
from collections import Counter

# Переменные, необходимые для классификации
J83_VARIABLES = [f"j83_{i}" for i in range(1, 9)]

CHECK_COLS = [
    "year",
    "adult",
    "j77",
    "j1",
    "j81",
    "j82",
    "j136",
    "j84_1",
    "j84_2"
] + J83_VARIABLES

# Размер одного блока
CHUNK_SIZE = 20000

# Проверяем наличие всех необходимых переменных
_, meta = pyreadstat.read_dta(
    str(DATA_PATH),
    metadataonly=True,
    encoding="utf-8"
)

missing_cols = [
    col for col in CHECK_COLS
    if col not in meta.column_names
]

if missing_cols:
    print("Отсутствуют переменные:", missing_cols)
else:
    print("Все необходимые переменные найдены.")

Все необходимые переменные найдены.


In [27]:
# Счётчики результатов классификации
classification_counts = Counter()
reason_counts = Counter()
yearly_counts = Counter()
yearly_reason_counts = Counter()

adult_processed = 0
row_offset = 0

while row_offset < meta.number_rows:

    # Читаем очередную часть файла
    chunk, _ = pyreadstat.read_dta(
        str(DATA_PATH),
        usecols=CHECK_COLS,
        row_offset=row_offset,
        row_limit=CHUNK_SIZE,
        encoding="utf-8"
    )

    if len(chunk) == 0:
        break

    # Важно: сохраняем исходный размер блока
    raw_chunk_size = len(chunk)

    # Оставляем только взрослых респондентов
    chunk = chunk[chunk["adult"] == 1].copy()

    adult_processed += len(chunk)

    # Классификация наблюдений
    for row in chunk.itertuples(index=False, name=None):

        data = dict(zip(CHECK_COLS, row))

        j83_values = [
            data[var] for var in J83_VARIABLES
        ]

        category, reason = classify_ilo(
            j77=data["j77"],
            j1=data["j1"],
            j81=data["j81"],
            j82=data["j82"],
            j136=data["j136"],
            j84_1=data["j84_1"],
            j84_2=data["j84_2"],
            j83_values=j83_values
        )

        year = data["year"]

        classification_counts[category] += 1
        reason_counts[reason] += 1

        if pd.notna(year):
            yearly_counts[(int(year), category)] += 1
            yearly_reason_counts[(int(year), reason)] += 1

    # Смещаемся на размер исходного блока,
    # а не на число взрослых после фильтрации
    row_offset += raw_chunk_size

    print(
        f"Обработано строк: {row_offset} из {meta.number_rows}"
    )

print("\nОбработка завершена.")
print("Количество взрослых наблюдений:", adult_processed)
print("Количество классифицированных наблюдений:",
      sum(classification_counts.values()))

Обработано строк: 20000 из 474600
Обработано строк: 40000 из 474600
Обработано строк: 60000 из 474600
Обработано строк: 80000 из 474600
Обработано строк: 100000 из 474600
Обработано строк: 120000 из 474600
Обработано строк: 140000 из 474600
Обработано строк: 160000 из 474600
Обработано строк: 180000 из 474600
Обработано строк: 200000 из 474600
Обработано строк: 220000 из 474600
Обработано строк: 240000 из 474600
Обработано строк: 260000 из 474600
Обработано строк: 280000 из 474600
Обработано строк: 300000 из 474600
Обработано строк: 320000 из 474600
Обработано строк: 340000 из 474600
Обработано строк: 360000 из 474600
Обработано строк: 380000 из 474600
Обработано строк: 400000 из 474600
Обработано строк: 420000 из 474600
Обработано строк: 440000 из 474600
Обработано строк: 460000 из 474600
Обработано строк: 474600 из 474600

Обработка завершена.
Количество взрослых наблюдений: 393382
Количество классифицированных наблюдений: 393382


In [28]:
classification_table = pd.DataFrame({
    "Категория": list(classification_counts.keys()),
    "Количество": list(classification_counts.values())
})

classification_table["Доля, %"] = (
    classification_table["Количество"]
    / classification_table["Количество"].sum() * 100
)

classification_table = classification_table.sort_values(
    "Категория"
).reset_index(drop=True)

display(classification_table)

,Категория,Количество,"Доля, %"
0,E,204072,51.876293
1,I,1,0.000254
2,NA,189309,48.123453


In [29]:
yearly_table = pd.DataFrame(
    [
        {
            "Год": year,
            "Категория": category,
            "Количество": count
        }
        for (year, category), count in yearly_counts.items()
    ]
)

yearly_table = yearly_table.sort_values(
    ["Год", "Категория"]
).reset_index(drop=True)

display(yearly_table)

,Год,Категория,Количество
0,1994,E,4616
1,1994,NA,4277
2,1995,E,4314
3,1995,NA,4103
4,1996,E,4156
...,...,...,...
56,2024,E,7590
57,2024,NA,6582
58,2025,E,7521
59,2025,I,1


In [30]:
yearly_pivot = yearly_table.pivot(
    index="Год",
    columns="Категория",
    values="Количество"
).fillna(0).astype(int)

display(yearly_pivot)

Категория,E,I,NA
Год,,,
1994,4616,0,4277
1995,4314,0,4103
1996,4156,0,4185
1998,4034,0,4665
2000,4337,0,4737
2001,4871,0,5227
2002,5102,0,5397
2003,5282,0,5354
2004,5339,0,5323


In [31]:
classification_table.to_csv(
    TABLES_DIR / "ilo_classification_total.csv",
    index=False,
    encoding="utf-8-sig"
)

yearly_table.to_csv(
    TABLES_DIR / "ilo_classification_by_year.csv",
    index=False,
    encoding="utf-8-sig"
)

yearly_pivot.to_csv(
    TABLES_DIR / "ilo_classification_pivot.csv",
    encoding="utf-8-sig"
)

reason_table = pd.DataFrame(
    [
        {"Причина": reason, "Количество": count}
        for reason, count in reason_counts.items()
    ]
).sort_values("Количество", ascending=False)

reason_table.to_csv(
    TABLES_DIR / "ilo_classification_reasons.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Все таблицы сохранены в results/tables.")

Все таблицы сохранены в results/tables.


In [32]:
import inspect

print(inspect.getsource(get_employment_status))

def get_employment_status(j77, j1):
    """
    Определение наличия работы.

    Возвращает:
    employed, quality, reason
    """

    if not is_missing(j77):

        if j77 == 1:
            return True, "high", "j77 = 1: есть работа"

        if j77 == 2:
            if not is_missing(j1) and j1 == 1:
                return None, "low", (
                    "Противоречие: j77 = 2, j1 = 1"
                )

            return False, "high", "j77 = 2: работы нет"

        return None, "low", "Специальный код j77"

    # Если j77 отсутствует, используем j1
    if not is_missing(j1):

        if j1 == 1:
            return True, "medium", (
                "j77 отсутствует, j1 = 1: указана работа"
            )

        if j1 == 5:
            return False, "medium", (
                "j77 отсутствует, j1 = 5: указано отсутствие работы"
            )

        return None, "low", (
            "По j1 нельзя однозначно определить занятость"
        )

    return None, "low", "Нет данны

In [33]:
test_employment = [
    (1, 1),
    (2, 5),
    (2, 1),
    (2, 2),
    (np.nan, 1),
    (np.nan, 5),
    (np.nan, 2),
    (np.nan, np.nan)
]

for j77, j1 in test_employment:
    print(
        f"j77={j77}, j1={j1} → "
        f"{get_employment_status(j77, j1)}"
    )

j77=1, j1=1 → (True, 'high', 'j77 = 1: есть работа')
j77=2, j1=5 → (False, 'high', 'j77 = 2: работы нет')
j77=2, j1=1 → (None, 'low', 'Противоречие: j77 = 2, j1 = 1')
j77=2, j1=2 → (False, 'high', 'j77 = 2: работы нет')
j77=nan, j1=1 → (True, 'medium', 'j77 отсутствует, j1 = 1: указана работа')
j77=nan, j1=5 → (False, 'medium', 'j77 отсутствует, j1 = 5: указано отсутствие работы')
j77=nan, j1=2 → (None, 'low', 'По j1 нельзя однозначно определить занятость')
j77=nan, j1=nan → (None, 'low', 'Нет данных о занятости')


In [34]:
employment_check = Counter()
row_offset = 0

while row_offset < meta.number_rows:

    chunk, _ = pyreadstat.read_dta(
        str(DATA_PATH),
        usecols=["adult", "j77", "j1"],
        row_offset=row_offset,
        row_limit=CHUNK_SIZE,
        encoding="utf-8"
    )

    if len(chunk) == 0:
        break

    raw_chunk_size = len(chunk)

    chunk = chunk[chunk["adult"] == 1]

    for j77, j1 in zip(chunk["j77"], chunk["j1"]):
        employment_check[(j77, j1)] += 1

    row_offset += raw_chunk_size

employment_check_table = pd.DataFrame(
    [
        {
            "j77": j77,
            "j1": j1,
            "Количество": count
        }
        for (j77, j1), count in employment_check.items()
    ]
).sort_values("Количество", ascending=False)

display(employment_check_table.head(30))

,j77,j1,Количество
0,1.0,1,196550
1,2.0,5,171137
8,NaN,5,9325
18,NaN,1,7521
3,1.0,2,6879
2,1.0,3,1033
4,1.0,4,409
6,2.0,99999997,181
19,NaN,2,169
7,2.0,99999998,94


In [35]:
import inspect

print(inspect.getsource(classify_ilo))

def classify_ilo(
    j77, j1, j81, j82, j136,
    j84_1, j84_2, j83_values
):
    """
    Классификация респондента по категориям E/U/I.

    E  — занятый;
    U  — безработный;
    I  — вне рабочей силы;
    NA — недостаточно информации для классификации.
    """

    # 1. Определяем наличие работы
    employment_result = get_employment_status(j77, j1)

    employed = employment_result[0]
    confidence = employment_result[1]
    if employed is True:
        return "E", "employed"

    if employed is None:
        return "NA", "employment_unknown"

    # 2. Учитываем ситуацию, когда работа уже найдена
    if j84_2 == 99999995:
        return "NA", "future_job"

    # 3. Проверяем активный поиск работы
    if j82 == 1:

        active_search_verified = assess_active_search(j83_values)

        # Нет подтверждения активного поиска
        if active_search_verified is not True:
            return "NA", "search_not_verified"

        # 4. Проверяем готовность приступить к работе
        

In [36]:
# Проверяем распределение результатов get_employment_status
employment_status_counts = Counter()

for (j77, j1), count in employment_check.items():

    result = get_employment_status(j77, j1)

    employment_status_counts[result[0]] += count

print("Распределение по статусу занятости:")

for status, count in employment_status_counts.items():
    print(status, count)

Распределение по статусу занятости:
True 212403
False 180751
None 228


In [37]:
# Проверяем согласованность функций на 20 000 строках

sample, _ = pyreadstat.read_dta(
    str(DATA_PATH),
    usecols=CHECK_COLS,
    row_limit=20000,
    encoding="utf-8"
)

sample = sample[sample["adult"] == 1].copy()

check_results = Counter()
check_employment = Counter()

for row in sample.itertuples(index=False, name=None):

    data = dict(zip(CHECK_COLS, row))

    employment_result = get_employment_status(
        data["j77"],
        data["j1"]
    )

    check_employment[employment_result[0]] += 1

    category, reason = classify_ilo(
        j77=data["j77"],
        j1=data["j1"],
        j81=data["j81"],
        j82=data["j82"],
        j136=data["j136"],
        j84_1=data["j84_1"],
        j84_2=data["j84_2"],
        j83_values=[data[var] for var in J83_VARIABLES]
    )

    check_results[reason] += 1

print("Проверка занятости:")
print(check_employment)

print("\nПричины классификации:")
print(check_results)

Проверка занятости:
Counter({True: 8062, None: 7643})

Причины классификации:
Counter({'employed': 8062, 'employment_unknown': 7643})


In [38]:
# Проверяем сочетания j77 и j1 в тестовой выборке

display(
    sample.groupby(
        ["year", "j77", "j1"],
        dropna=False
    ).size().reset_index(name="Количество")
)

,year,j77,j1,Количество
0,1994,1,1,4616
1,1994,1,2,181
2,1994,1,3,55
3,1994,1,4,42
4,1994,2,5,3993
5,1994,2,99999997,2
6,1994,2,99999998,1
7,1994,2,99999999,3
8,1995,1,1,3446
9,1995,1,2,154


In [39]:
# Проверяем распределение ответов j1
# среди респондентов, у которых j77 отсутствует

display(
    sample[sample["j77"].isna()]
    .groupby(["year", "j1"], dropna=False)
    .size()
    .reset_index(name="Количество")
)

,year,j1,Количество


In [40]:
print("Распределение j77:")
display(
    sample["j77"].value_counts(dropna=False).sort_index()
)

Распределение j77:


j77
1    8561
2    7144
Name: count, dtype: int64

In [41]:
unknown_details = Counter()

for j77, j1 in zip(sample["j77"], sample["j1"]):

    result = get_employment_status(j77, j1)

    if result[0] is None:
        unknown_details[result[2]] += 1

for reason, count in unknown_details.items():
    print(reason, "→", count)

In [42]:
print("Распределение j1:")
display(
    sample["j1"].value_counts(dropna=False).sort_index()
)

Распределение j1:


j1
1           8062
2            335
3             95
4             69
5           7131
99999997       8
99999998       2
99999999       3
Name: count, dtype: int64

In [43]:
print("Результат функции для j77 = 2:")

for value in sorted(sample.loc[sample["j77"] == 2, "j1"].unique()):
    print(
        f"j1 = {value} →",
        get_employment_status(2, value)
    )

Результат функции для j77 = 2:
j1 = 5 → (False, 'high', 'j77 = 2: работы нет')
j1 = 99999997 → (False, 'high', 'j77 = 2: работы нет')
j1 = 99999998 → (False, 'high', 'j77 = 2: работы нет')
j1 = 99999999 → (False, 'high', 'j77 = 2: работы нет')


In [44]:
# Проверяем результат функции для каждой комбинации j77 и j1

combination_check = []

for (j77, j1), group in sample.groupby(
    ["j77", "j1"],
    dropna=False
):
    result = get_employment_status(j77, j1)

    combination_check.append({
        "j77": j77,
        "j1": j1,
        "Количество": len(group),
        "Результат функции": result[0],
        "Причина": result[2]
    })

combination_check = pd.DataFrame(combination_check)

display(
    combination_check.sort_values(
        ["j77", "j1"],
        na_position="last"
    )
)

,j77,j1,Количество,Результат функции,Причина
0,1,1,8062,True,j77 = 1: есть работа
1,1,2,335,True,j77 = 1: есть работа
2,1,3,95,True,j77 = 1: есть работа
3,1,4,69,True,j77 = 1: есть работа
4,2,5,7131,False,j77 = 2: работы нет
5,2,99999997,8,False,j77 = 2: работы нет
6,2,99999998,2,False,j77 = 2: работы нет
7,2,99999999,3,False,j77 = 2: работы нет


In [45]:
print("Сумма по результатам функции:")

display(
    combination_check.groupby(
        "Результат функции",
        dropna=False
    )["Количество"].sum()
)

Сумма по результатам функции:


Результат функции
False    7144
True     8561
Name: Количество, dtype: int64

In [46]:
import inspect

print(inspect.getsource(classify_ilo))

def classify_ilo(
    j77, j1, j81, j82, j136,
    j84_1, j84_2, j83_values
):
    """
    Классификация респондента по категориям E/U/I.

    E  — занятый;
    U  — безработный;
    I  — вне рабочей силы;
    NA — недостаточно информации для классификации.
    """

    # 1. Определяем наличие работы
    employment_result = get_employment_status(j77, j1)

    employed = employment_result[0]
    confidence = employment_result[1]
    if employed is True:
        return "E", "employed"

    if employed is None:
        return "NA", "employment_unknown"

    # 2. Учитываем ситуацию, когда работа уже найдена
    if j84_2 == 99999995:
        return "NA", "future_job"

    # 3. Проверяем активный поиск работы
    if j82 == 1:

        active_search_verified = assess_active_search(j83_values)

        # Нет подтверждения активного поиска
        if active_search_verified is not True:
            return "NA", "search_not_verified"

        # 4. Проверяем готовность приступить к работе
        

In [49]:
sample_results = Counter()

for row in sample.itertuples(index=False, name=None):

    # Используем фактический порядок столбцов DataFrame
    data = dict(zip(sample.columns, row))

    category, reason = classify_ilo(
        j77=data["j77"],
        j1=data["j1"],
        j81=data["j81"],
        j82=data["j82"],
        j136=data["j136"],
        j84_1=data["j84_1"],
        j84_2=data["j84_2"],
        j83_values=[data[var] for var in J83_VARIABLES]
    )

    sample_results[reason] += 1

print(sample_results)

Counter({'employed': 8561, 'insufficient_information': 4765, 'wants_job_availability_unknown': 1432, 'availability_missing': 914, 'search_not_verified': 33})


In [48]:
print("Проверка непосредственно:")
print("j77=1, j1=2:", get_employment_status(1, 2))
print("j77=2, j1=5:", get_employment_status(2, 5))

print("\nПроверка классификатора:")
print(
    classify_ilo(
        j77=1,
        j1=2,
        j81=1,
        j82=1,
        j136=1,
        j84_1=1,
        j84_2=np.nan,
        j83_values=[1, 2, 2, 2, 2, 2, 2, 2]
    )
)

print(
    classify_ilo(
        j77=2,
        j1=5,
        j81=1,
        j82=1,
        j136=1,
        j84_1=1,
        j84_2=np.nan,
        j83_values=[1, 2, 2, 2, 2, 2, 2, 2]
    )
)

Проверка непосредственно:
j77=1, j1=2: (True, 'high', 'j77 = 1: есть работа')
j77=2, j1=5: (False, 'high', 'j77 = 2: работы нет')

Проверка классификатора:
('E', 'employed')
('U', 'unemployed')


In [50]:

# ==========================================
# ПОЛНАЯ КЛАССИФИКАЦИЯ РЕСПОНДЕНТОВ RLMS-HSE
# ==========================================

from collections import Counter
import pandas as pd
import numpy as np
import pyreadstat

# 1. Пути и параметры

DATA_PATH = DATA_DIR / "RLMS_IND_1994_2025_rus.dta"

CHUNK_SIZE = 20000

J83_VARIABLES = [f"j83_{i}" for i in range(1, 9)]

CHECK_COLS = [
    "year", "adult", "j77", "j1",
    "j81", "j82", "j136",
    "j84_1", "j84_2"
] + J83_VARIABLES

# 2. Загружаем метаданные

_, meta = pyreadstat.read_dta(
    str(DATA_PATH),
    metadataonly=True,
    encoding="utf-8"
)

# 3. Проверяем наличие переменных

missing_cols = [
    col for col in CHECK_COLS
    if col not in meta.column_names
]

if missing_cols:
    raise ValueError(
        f"В файле отсутствуют переменные: {missing_cols}"
    )

# 4. Обнуляем счётчики

classification_counts = Counter()
reason_counts = Counter()

yearly_counts = Counter()
yearly_reason_counts = Counter()

adult_processed = 0
row_offset = 0

# 5. Читаем и обрабатываем файл частями

while row_offset < meta.number_rows:

    chunk, _ = pyreadstat.read_dta(
        str(DATA_PATH),
        usecols=CHECK_COLS,
        row_offset=row_offset,
        row_limit=CHUNK_SIZE,
        encoding="utf-8"
    )

    if len(chunk) == 0:
        break

    # Размер исходного блока до фильтрации
    raw_chunk_size = len(chunk)

    # Оставляем взрослых респондентов
    chunk = chunk[chunk["adult"] == 1].copy()

    adult_processed += len(chunk)

    # Классифицируем каждого респондента
    for row in chunk.itertuples(index=False, name=None):

        # Используем фактический порядок столбцов
        data = dict(zip(chunk.columns, row))

        j83_values = [
            data[var] for var in J83_VARIABLES
        ]

        category, reason = classify_ilo(
            j77=data["j77"],
            j1=data["j1"],
            j81=data["j81"],
            j82=data["j82"],
            j136=data["j136"],
            j84_1=data["j84_1"],
            j84_2=data["j84_2"],
            j83_values=j83_values
        )

        year = data["year"]

        classification_counts[category] += 1
        reason_counts[reason] += 1

        if pd.notna(year):
            year = int(year)

            yearly_counts[(year, category)] += 1
            yearly_reason_counts[(year, reason)] += 1

    # Переходим к следующему исходному блоку
    row_offset += raw_chunk_size

    print(
        f"Обработано {row_offset} из {meta.number_rows} строк"
    )

# 6. Проверяем итог обработки

classified_total = sum(classification_counts.values())

print("\nОБРАБОТКА ЗАВЕРШЕНА")
print("Всего взрослых наблюдений:", adult_processed)
print("Всего классифицированных:", classified_total)

print("\nРаспределение категорий:")
print(classification_counts)

print("\nРаспределение причин:")
print(reason_counts)

assert adult_processed == classified_total, (
    "Количество обработанных наблюдений не совпадает!"
)


Обработано 20000 из 474600 строк
Обработано 40000 из 474600 строк
Обработано 60000 из 474600 строк
Обработано 80000 из 474600 строк
Обработано 100000 из 474600 строк
Обработано 120000 из 474600 строк
Обработано 140000 из 474600 строк
Обработано 160000 из 474600 строк
Обработано 180000 из 474600 строк
Обработано 200000 из 474600 строк
Обработано 220000 из 474600 строк
Обработано 240000 из 474600 строк
Обработано 260000 из 474600 строк
Обработано 280000 из 474600 строк
Обработано 300000 из 474600 строк
Обработано 320000 из 474600 строк
Обработано 340000 из 474600 строк
Обработано 360000 из 474600 строк
Обработано 380000 из 474600 строк
Обработано 400000 из 474600 строк
Обработано 420000 из 474600 строк
Обработано 440000 из 474600 строк
Обработано 460000 из 474600 строк
Обработано 474600 из 474600 строк

ОБРАБОТКА ЗАВЕРШЕНА
Всего взрослых наблюдений: 393382
Всего классифицированных: 393382

Распределение категорий:
Counter({'E': 212403, 'NA': 153170, 'I': 16154, 'U': 11655})

Распределени

In [51]:
# Сохраняем финальные результаты классификации

classification_table.to_csv(
    TABLES_DIR / "ilo_classification_total_final_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

yearly_table.to_csv(
    TABLES_DIR / "ilo_classification_by_year_final_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

yearly_pivot.to_csv(
    TABLES_DIR / "ilo_classification_pivot_final_v1.csv",
    encoding="utf-8-sig"
)

reason_table.to_csv(
    TABLES_DIR / "ilo_classification_reasons_final_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Финальные результаты сохранены.")

Финальные результаты сохранены.


In [52]:
# Проверяем структуру неопределённых наблюдений

na_reasons = reason_table[
    reason_table["Причина"] != "employed"
].copy()

na_reasons["Доля среди NA, %"] = (
    na_reasons["Количество"] / 153170 * 100
)

display(
    na_reasons.sort_values(
        "Количество",
        ascending=False
    )
)

,Причина,Количество,"Доля среди NA, %"
1,employment_unknown,189140,123.483711
2,insufficient_information,169,0.110335
3,wants_job_availability_unknown,1,0.000653


In [53]:
insufficient_by_year = pd.DataFrame([
    {
        "Год": year,
        "Количество": count
    }
    for (year, reason), count in yearly_reason_counts.items()
    if reason == "insufficient_information"
])

display(
    insufficient_by_year.sort_values("Год")
)

,Год,Количество
0,1994,2606
1,1995,2590
2,1996,2556
3,1998,2866
4,2000,3023
5,2001,3361
6,2002,3553
7,2003,3589
8,2004,3566
9,2005,3583


In [54]:
# Формируем актуальную таблицу причин NA

NA_REASONS = [
    "insufficient_information",
    "availability_missing",
    "search_not_verified",
    "availability_unknown",
    "employment_unknown",
    "future_job"
]

na_reasons = pd.DataFrame([
    {
        "Причина": reason,
        "Количество": reason_counts.get(reason, 0)
    }
    for reason in NA_REASONS
])

na_reasons["Доля среди NA, %"] = (
    na_reasons["Количество"] / classification_counts["NA"] * 100
)

na_reasons = na_reasons.sort_values(
    "Количество",
    ascending=False
).reset_index(drop=True)

display(na_reasons)

print("Сумма причин NA:", na_reasons["Количество"].sum())
print("Всего NA:", classification_counts["NA"])

assert na_reasons["Количество"].sum() == classification_counts["NA"]

,Причина,Количество,"Доля среди NA, %"
0,insufficient_information,149480,97.590912
1,availability_missing,1883,1.229353
2,search_not_verified,734,0.479206
3,availability_unknown,626,0.408696
4,employment_unknown,228,0.148854
5,future_job,219,0.142978


Сумма причин NA: 153170
Всего NA: 153170


In [55]:
na_reasons.to_csv(
    TABLES_DIR / "ilo_classification_NA_breakdown_final_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Актуальная таблица причин NA сохранена.")

Актуальная таблица причин NA сохранена.


In [56]:
print(
    "Сумма по годам:",
    insufficient_by_year["Количество"].sum()
)

print(
    "Общее количество:",
    reason_counts["insufficient_information"]
)

Сумма по годам: 149480
Общее количество: 149480


In [57]:
# Анализ причин insufficient_information
# по годам и сочетаниям ответов

insufficient_combinations = Counter()

row_offset = 0

while row_offset < meta.number_rows:

    chunk, _ = pyreadstat.read_dta(
        str(DATA_PATH),
        usecols=CHECK_COLS,
        row_offset=row_offset,
        row_limit=CHUNK_SIZE,
        encoding="utf-8"
    )

    if len(chunk) == 0:
        break

    raw_chunk_size = len(chunk)

    chunk = chunk[chunk["adult"] == 1]

    for row in chunk.itertuples(index=False, name=None):

        data = dict(zip(chunk.columns, row))

        category, reason = classify_ilo(
            j77=data["j77"],
            j1=data["j1"],
            j81=data["j81"],
            j82=data["j82"],
            j136=data["j136"],
            j84_1=data["j84_1"],
            j84_2=data["j84_2"],
            j83_values=[
                data[var] for var in J83_VARIABLES
            ]
        )

        if reason == "insufficient_information":

            key = (
                data["year"],
                data["j81"],
                data["j82"],
                data["j84_1"],
                data["j84_2"]
            )

            insufficient_combinations[key] += 1

    row_offset += raw_chunk_size

    print(f"Обработано строк: {row_offset}")

# Формируем таблицу сочетаний

insufficient_table = pd.DataFrame([
    {
        "Год": key[0],
        "j81": key[1],
        "j82": key[2],
        "j84_1": key[3],
        "j84_2": key[4],
        "Количество": count
    }
    for key, count in insufficient_combinations.items()
])

insufficient_table = insufficient_table.sort_values(
    "Количество",
    ascending=False
).reset_index(drop=True)

display(insufficient_table.head(30))

print(
    "Всего наблюдений:",
    insufficient_table["Количество"].sum()
)

Обработано строк: 20000
Обработано строк: 40000
Обработано строк: 60000
Обработано строк: 80000
Обработано строк: 100000
Обработано строк: 120000
Обработано строк: 140000
Обработано строк: 160000
Обработано строк: 180000
Обработано строк: 200000
Обработано строк: 220000
Обработано строк: 240000
Обработано строк: 260000
Обработано строк: 280000
Обработано строк: 300000
Обработано строк: 320000
Обработано строк: 340000
Обработано строк: 360000
Обработано строк: 380000
Обработано строк: 400000
Обработано строк: 420000
Обработано строк: 440000
Обработано строк: 460000
Обработано строк: 474600


,Год,j81,j82,j84_1,j84_2,Количество
0,2012,2,NaN,NaN,NaN,6331
1,2013,2,NaN,NaN,NaN,6306
2,2011,2,NaN,NaN,NaN,6077
3,2021,2,NaN,NaN,NaN,6012
4,2020,2,NaN,NaN,NaN,5996
5,2019,2,NaN,NaN,NaN,5940
6,2017,2,NaN,NaN,NaN,5889
7,2022,2,NaN,NaN,NaN,5790
8,2010,2,NaN,NaN,NaN,5784
9,2018,2,NaN,NaN,NaN,5741


Всего наблюдений: 149480


In [58]:

def classify_ilo(
    j77, j1, j81, j82, j136,
    j84_1, j84_2, j83_values
):
    """
    Предварительная классификация по категориям E/U/I.

    E  — занятый;
    U  — безработный;
    I  — вне рабочей силы;
    NA — недостаточно информации.
    """

    # 1. Определяем наличие работы
    employment_result = get_employment_status(j77, j1)
    employed = employment_result[0]

    if employed is True:
        return "E", "employed"

    if employed is None:
        return "NA", "employment_unknown"

    # 2. Проверяем, хочет ли человек искать работу
    # j81 = 2: не хочет искать работу
    if j81 == 2:
        return "I", "outside_labour_force"

    # 3. Если ответ о желании искать работу отсутствует
    if j81 != 1:
        return "NA", "insufficient_information"

    # 4. Проверяем ситуацию, когда работа уже найдена
    if j84_2 == 99999995:
        return "NA", "future_job"

    # 5. Проверяем фактический поиск работы
    if j82 == 1:

        active_search_verified = assess_active_search(j83_values)

        if active_search_verified is not True:
            return "NA", "search_not_verified"

        # 6. Проверяем готовность приступить к работе
        if j84_1 == 1:
            return "U", "unemployed"

        if j84_1 == 2:
            return "NA", "availability_unknown"

        return "NA", "availability_missing"

    # 7. Человек хочет искать работу, но не искал
    if j82 == 2:
        if j84_1 == 1:
            return "I", "potential_labour_force_candidate"

        return "I", "wants_job_availability_unknown"

    # 8. Недостаточно информации
    return "NA", "insufficient_information"


In [59]:
# Проверяем новую логику маршрутизации

test_cases = [
    ("Не хочет искать, j82 пропущен", 2, np.nan),
    ("Не хочет искать, j82 = 2", 2, 2),
    ("Хочет искать, но не искал", 1, 2),
    ("Хочет искать и искал", 1, 1)
]

for name, j81, j82 in test_cases:

    result = classify_ilo(
        j77=2,
        j1=5,
        j81=j81,
        j82=j82,
        j136=np.nan,
        j84_1=1,
        j84_2=np.nan,
        j83_values=[np.nan] * 8
    )

    print(name, "→", result)

Не хочет искать, j82 пропущен → ('I', 'outside_labour_force')
Не хочет искать, j82 = 2 → ('I', 'outside_labour_force')
Хочет искать, но не искал → ('I', 'potential_labour_force_candidate')
Хочет искать и искал → ('NA', 'search_not_verified')


In [60]:
# Проверяем случай подтверждённого активного поиска

result = classify_ilo(
    j77=2,
    j1=5,
    j81=1,
    j82=1,
    j136=1,
    j84_1=1,
    j84_2=np.nan,
    j83_values=[1, 2, 2, 2, 2, 2, 2, 2]
)

print(result)

('U', 'unemployed')


In [61]:
classification_table.to_csv(
    TABLES_DIR / "ilo_classification_total_final_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

yearly_table.to_csv(
    TABLES_DIR / "ilo_classification_by_year_final_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

yearly_pivot.to_csv(
    TABLES_DIR / "ilo_classification_pivot_final_v2.csv",
    encoding="utf-8-sig"
)

reason_table.to_csv(
    TABLES_DIR / "ilo_classification_reasons_final_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

In [62]:
import pandas as pd
import numpy as np
import pyreadstat
from pathlib import Path
from collections import Counter

# Пути к данным и результатам
DATA_DIR = Path(r"D:\RLMS_VKR")
DATA_PATH = DATA_DIR / "data" / "RLMS_IND_1994_2025_rus.dta"
TABLES_DIR = DATA_DIR / "results" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Размер одного блока при чтении большого файла
CHUNK_SIZE = 20000

# Переменные, необходимые для классификации
CHECK_COLS = [
    "year", "adult", "j77", "j1",
    "j81", "j82", "j136",
    "j84_1", "j84_2"
] + [f"j83_{i}" for i in range(1, 9)]


def is_missing(value):
    return pd.isna(value)


def get_employment_status(j77, j1):
    """
    Определение наличия работы.

    Возвращает:
    employed — True, False или None;
    quality — оценка надёжности;
    reason — причина решения.
    """

    if not is_missing(j77):
        if j77 == 1:
            return True, "high", "j77 = 1: есть работа"

        if j77 == 2:
            if not is_missing(j1) and j1 == 1:
                return None, "low", "Противоречие: j77 = 2, j1 = 1"

            return False, "high", "j77 = 2: работы нет"

        return None, "low", "Неизвестный код j77"

    # Если j77 отсутствует, используем j1
    if not is_missing(j1):
        if j1 == 1:
            return True, "medium", "j77 отсутствует, j1 = 1"

        if j1 == 5:
            return False, "medium", "j77 отсутствует, j1 = 5"

    return None, "low", "Недостаточно данных о занятости"


# Способы поиска работы
J83_ACTIVE = [
    "j83_1", "j83_2", "j83_3", "j83_4", "j83_5"
]

J83_AMBIGUOUS = [
    "j83_6", "j83_7", "j83_8"
]


def get_search_methods(j83_values):
    """Возвращает способы поиска работы, отмеченные ответом 1."""

    return [
        name
        for name, value in j83_values.items()
        if not is_missing(value) and value == 1
    ]


def assess_active_search(j83_values):
    """
    Консервативная операционализация активного поиска работы.

    True — отмечен хотя бы один однозначный способ поиска;
    False — все восемь способов заполнены и ни один не отмечен;
    None — информации недостаточно или отмечены только неоднозначные способы.
    """

    # Есть хотя бы один однозначный способ поиска
    for name in J83_ACTIVE:
        value = j83_values.get(name)

        if not is_missing(value) and value == 1:
            return True

    # Проверяем, есть ли отмеченные неоднозначные способы
    ambiguous_yes = any(
        not is_missing(j83_values.get(name))
        and j83_values.get(name) == 1
        for name in J83_AMBIGUOUS
    )

    if ambiguous_yes:
        return None

    # Все восемь ответов должны быть известны
    all_answers_known = all(
        not is_missing(j83_values.get(f"j83_{i}"))
        and j83_values.get(f"j83_{i}") in (1, 2)
        for i in range(1, 9)
    )

    if all_answers_known:
        return False

    return None


def classify_ilo(
    j77, j1, j81, j82, j136,
    j84_1, j84_2, j83_values
):
    """
    Предварительная классификация наблюдений на E, U, I и NA.

    E  — занятые;
    U  — безработные;
    I  — вне рабочей силы;
    NA — недостаточно информации для классификации.
    """

    # 1. Определяем занятость
    employment_result = get_employment_status(j77, j1)
    employed = employment_result[0]

    if employed is True:
        return "E", "employed"

    if employed is None:
        return "NA", "employment_unknown"

    # 2. Человек не занят и не хочет искать работу
    # j82 может отсутствовать из-за маршрутизации анкеты
    if not is_missing(j81) and j81 == 2:
        return "I", "outside_labour_force"

    # 3. Если желание искать работу неизвестно
    if is_missing(j81) or j81 != 1:
        return "NA", "insufficient_information"

    # 4. Проверяем специальный код j84_2
    if not is_missing(j84_2) and j84_2 == 99999995:
        return "NA", "future_job"

    # 5. Человек хочет найти работу и искал её за последние 30 дней
    if not is_missing(j82) and j82 == 1:

        active_search_verified = assess_active_search(j83_values)

        if active_search_verified is not True:
            return "NA", "search_not_verified"

        if not is_missing(j84_1) and j84_1 == 1:
            return "U", "unemployed"

        if not is_missing(j84_1) and j84_1 == 2:
            return "NA", "availability_unknown"

        return "NA", "availability_missing"

    # 6. Хочет найти работу, но не искал её за последние 30 дней
    if not is_missing(j82) and j82 == 2:

        if not is_missing(j84_1) and j84_1 == 1:
            return "I", "potential_labour_force_candidate"

        return "I", "wants_job_availability_unknown"

    return "NA", "insufficient_information"


print("Функции классификации v2 успешно определены.")

Функции классификации v2 успешно определены.


In [64]:
# Обнуляем счётчики
status_counts = Counter()
reason_counts = Counter()
yearly_counts = Counter()

total_adults = 0
processed_rows = 0

# Сначала получаем количество строк в файле
_, file_meta = pyreadstat.read_dta(
    str(DATA_PATH),
    metadataonly=True,
    encoding="utf-8"
)

total_rows = file_meta.number_rows

print(f"Всего строк в файле: {total_rows:,}")

# Читаем файл вручную частями
for offset in range(0, total_rows, CHUNK_SIZE):

    chunk = pyreadstat.read_dta(
        str(DATA_PATH),
        usecols=CHECK_COLS,
        encoding="utf-8",
        row_offset=offset,
        row_limit=CHUNK_SIZE
    )[0]

    raw_chunk_size = len(chunk)

    # Оставляем только взрослых
    chunk = chunk.loc[chunk["adult"] == 1]

    total_adults += len(chunk)

    # Обрабатываем каждое наблюдение
    for row in chunk.itertuples(index=False, name=None):

        # ВАЖНО: сопоставляем значения с реальными названиями столбцов
        data = dict(zip(chunk.columns, row))

        j83_values = {
            f"j83_{i}": data.get(f"j83_{i}")
            for i in range(1, 9)
        }

        status, reason = classify_ilo(
            j77=data.get("j77"),
            j1=data.get("j1"),
            j81=data.get("j81"),
            j82=data.get("j82"),
            j136=data.get("j136"),
            j84_1=data.get("j84_1"),
            j84_2=data.get("j84_2"),
            j83_values=j83_values
        )

        year = data.get("year")

        status_counts[status] += 1
        reason_counts[reason] += 1

        if not is_missing(year):
            yearly_counts[(int(year), status)] += 1

    processed_rows += raw_chunk_size

    print(
        f"Обработано: {processed_rows:,} из {total_rows:,} строк | "
        f"Взрослых: {total_adults:,}"
    )

print("\nОбработка завершена!")

print("Всего взрослых наблюдений:", total_adults)
print("Всего классифицированных наблюдений:", sum(status_counts.values()))

assert sum(status_counts.values()) == total_adults, (
    "Ошибка: количество классифицированных наблюдений не совпадает!"
)

Всего строк в файле: 474,600
Обработано: 20,000 из 474,600 строк | Взрослых: 15,705
Обработано: 40,000 из 474,600 строк | Взрослых: 31,780
Обработано: 60,000 из 474,600 строк | Взрослых: 48,326
Обработано: 80,000 из 474,600 строк | Взрослых: 65,091
Обработано: 100,000 из 474,600 строк | Взрослых: 81,905
Обработано: 120,000 из 474,600 строк | Взрослых: 98,841
Обработано: 140,000 из 474,600 строк | Взрослых: 115,824
Обработано: 160,000 из 474,600 строк | Взрослых: 132,727
Обработано: 180,000 из 474,600 строк | Взрослых: 149,508
Обработано: 200,000 из 474,600 строк | Взрослых: 166,206
Обработано: 220,000 из 474,600 строк | Взрослых: 182,838
Обработано: 240,000 из 474,600 строк | Взрослых: 199,461
Обработано: 260,000 из 474,600 строк | Взрослых: 216,002
Обработано: 280,000 из 474,600 строк | Взрослых: 232,497
Обработано: 300,000 из 474,600 строк | Взрослых: 248,847
Обработано: 320,000 из 474,600 строк | Взрослых: 265,207
Обработано: 340,000 из 474,600 строк | Взрослых: 281,521
Обработано: 

In [65]:
# 1. Общая таблица классификации
classification_table = pd.DataFrame({
    "Категория": ["E", "U", "I", "NA"],
    "Количество": [
        status_counts["E"],
        status_counts["U"],
        status_counts["I"],
        status_counts["NA"]
    ]
})

classification_table["Доля, %"] = (
    classification_table["Количество"] / total_adults * 100
).round(2)


# 2. Таблица причин классификации
reason_table = pd.DataFrame(
    [
        {"Причина": reason, "Количество": count}
        for reason, count in reason_counts.items()
    ]
).sort_values("Количество", ascending=False).reset_index(drop=True)


# 3. Таблица классификации по годам
yearly_table = pd.DataFrame(
    [
        {
            "Год": year,
            "Категория": status,
            "Количество": count
        }
        for (year, status), count in yearly_counts.items()
    ]
)

yearly_pivot = (
    yearly_table
    .pivot(index="Год", columns="Категория", values="Количество")
    .fillna(0)
    .astype(int)
)

# Обеспечиваем наличие всех четырёх категорий
yearly_pivot = yearly_pivot.reindex(
    columns=["E", "U", "I", "NA"],
    fill_value=0
)

yearly_pivot = yearly_pivot.reset_index()


# 4. Проверки итогов
assert classification_table["Количество"].sum() == total_adults

assert reason_table["Количество"].sum() == total_adults

assert yearly_pivot[["E", "U", "I", "NA"]].to_numpy().sum() == total_adults


classification_table.to_csv(
    TABLES_DIR / "ilo_classification_total_v2_new.csv",
    index=False, encoding="utf-8-sig"
)

reason_table.to_csv(
    TABLES_DIR / "ilo_classification_reasons_v2_new.csv",
    index=False, encoding="utf-8-sig"
)

yearly_table.to_csv(
    TABLES_DIR / "ilo_classification_by_year_v2_new.csv",
    index=False, encoding="utf-8-sig"
)

yearly_pivot.to_csv(
    TABLES_DIR / "ilo_classification_pivot_v2_new.csv",
    index=False, encoding="utf-8-sig"
)


# 6. Вывод результатов
print("ОБЩАЯ КЛАССИФИКАЦИЯ")
display(classification_table)

print("\nПРИЧИНЫ КЛАССИФИКАЦИИ")
display(reason_table)

print("\nКЛАССИФИКАЦИЯ ПО ГОДАМ")
display(yearly_pivot)

print("\nВсе таблицы сохранены в:", TABLES_DIR)

ОБЩАЯ КЛАССИФИКАЦИЯ


,Категория,Количество,"Доля, %"
0,E,212403,53.99
1,U,11655,2.96
2,I,155346,39.49
3,NA,13978,3.55



ПРИЧИНЫ КЛАССИФИКАЦИИ


,Причина,Количество
0,employed,212403
1,outside_labour_force,139192
2,unemployed,11655
3,insufficient_information,10288
4,potential_labour_force_candidate,8906
5,wants_job_availability_unknown,7248
6,availability_missing,1883
7,search_not_verified,734
8,availability_unknown,626
9,employment_unknown,228



КЛАССИФИКАЦИЯ ПО ГОДАМ


Категория,Год,E,U,I,NA
0,1994,4894,0,3411,588
1,1995,4571,0,3234,612
2,1996,4382,0,3287,672
3,1998,4247,635,3713,104
4,2000,4508,539,3900,127
5,2001,5089,553,4305,151
6,2002,5324,542,4489,144
7,2003,5506,513,4479,138
8,2004,5564,544,4353,201
9,2005,5426,452,4260,199



Все таблицы сохранены в: D:\RLMS_VKR\results\tables


In [66]:
import pandas as pd
import numpy as np
import pyreadstat

# Переменные для повторного чтения и анализа панели
PROJECT_COLS = [
    "idind", "year", "adult",
    "j77", "j1", "j81", "j82", "j136",
    "j84_1", "j84_2"
] + [f"j83_{i}" for i in range(1, 9)]

# Проверяем наличие идентификатора
_, meta = pyreadstat.read_dta(
    str(DATA_PATH),
    metadataonly=True,
    encoding="utf-8"
)

assert "idind" in meta.column_names, "В файле не найден idind!"

# Считываем количество строк
total_rows = meta.number_rows

records = []

print("Начинаем повторную обработку данных для анализа панели...")

for offset in range(0, total_rows, CHUNK_SIZE):

    chunk = pyreadstat.read_dta(
        str(DATA_PATH),
        usecols=PROJECT_COLS,
        encoding="utf-8",
        row_offset=offset,
        row_limit=CHUNK_SIZE
    )[0]

    chunk = chunk.loc[chunk["adult"] == 1]

    for row in chunk.itertuples(index=False, name=None):

        data = dict(zip(chunk.columns, row))

        j83_values = {
            f"j83_{i}": data.get(f"j83_{i}")
            for i in range(1, 9)
        }

        status, reason = classify_ilo(
            j77=data.get("j77"),
            j1=data.get("j1"),
            j81=data.get("j81"),
            j82=data.get("j82"),
            j136=data.get("j136"),
            j84_1=data.get("j84_1"),
            j84_2=data.get("j84_2"),
            j83_values=j83_values
        )

        records.append({
            "idind": data["idind"],
            "year": int(data["year"]),
            "status": status,
            "reason": reason
        })

    print(f"Обработано строк: {min(offset + CHUNK_SIZE, total_rows):,} / {total_rows:,}")

panel = pd.DataFrame(records)

print("\nОбработка завершена.")
print("Количество наблюдений:", len(panel))
print("Количество уникальных респондентов:", panel["idind"].nunique())

# Проверяем, что у одного человека нет нескольких записей за один год
duplicates = panel.duplicated(["idind", "year"]).sum()

print("Повторных записей idind-year:", duplicates)

if duplicates > 0:
    display(
        panel.loc[
            panel.duplicated(["idind", "year"], keep=False)
        ].sort_values(["idind", "year"]).head(20)
    )
    raise ValueError(
        "Есть повторные записи одного респондента за год. "
        "Нужно проверить их до расчёта переходов."
    )


# =====================================================
# 2.1. ЧАСТОТЫ И ДОЛИ E/U/I/NA ПО ГОДАМ
# =====================================================

yearly_counts = (
    panel.groupby(["year", "status"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["E", "U", "I", "NA"], fill_value=0)
)

yearly_shares = (
    yearly_counts.div(yearly_counts.sum(axis=1), axis=0) * 100
).round(2)

yearly_shares.columns = [
    f"{col}_доля_%" for col in yearly_shares.columns
]

yearly_final = yearly_counts.join(yearly_shares).reset_index()

yearly_final.to_csv(
    TABLES_DIR / "stage2_yearly_frequencies_shares.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n2.1. ЧАСТОТЫ И ДОЛИ ПО ГОДАМ")
display(yearly_final)


# =====================================================
# 2.2. МАТРИЦЫ ПЕРЕХОДОВ МЕЖДУ СОСЕДНИМИ ВОЛНАМИ
# =====================================================

# Для основного анализа используем волны начиная с 1998 года
transition_data = panel.loc[panel["year"] >= 1998].copy()

waves = sorted(transition_data["year"].unique())

transition_counts = {}
transition_percentages = {}
transition_details = []

for i in range(len(waves) - 1):

    year1 = waves[i]
    year2 = waves[i + 1]

    first = transition_data.loc[
        transition_data["year"] == year1,
        ["idind", "status"]
    ].rename(columns={"status": "from"})

    second = transition_data.loc[
        transition_data["year"] == year2,
        ["idind", "status"]
    ].rename(columns={"status": "to"})

    matched = first.merge(second, on="idind", how="inner")

    # Для матрицы переходов используем только E, U, I
    matched_valid = matched.loc[
        matched["from"].isin(["E", "U", "I"])
        & matched["to"].isin(["E", "U", "I"])
    ]

    matrix = pd.crosstab(
        matched_valid["from"],
        matched_valid["to"]
    ).reindex(
        index=["E", "U", "I"],
        columns=["E", "U", "I"],
        fill_value=0
    )

    # Проценты относительно исходного состояния
    matrix_percent = (
        matrix.div(matrix.sum(axis=1).replace(0, np.nan), axis=0) * 100
    ).round(2)

    label = f"{year1}_{year2}"

    transition_counts[label] = matrix
    transition_percentages[label] = matrix_percent

    for origin in ["E", "U", "I"]:
        for destination in ["E", "U", "I"]:

            transition_details.append({
                "Период": label,
                "Из": origin,
                "В": destination,
                "Количество": matrix.loc[origin, destination],
                "Доля_от_исходной_категории_%":
                    matrix_percent.loc[origin, destination]
            })

    print(f"\nПереходы {year1} → {year2}")
    print("Число респондентов с известными состояниями E/U/I:",
          len(matched_valid))
    print("\nКоличество переходов:")
    display(matrix)
    print("\nДоли переходов, %:")
    display(matrix_percent)


transition_details = pd.DataFrame(transition_details)

transition_details.to_csv(
    TABLES_DIR / "stage2_transition_matrices.csv",
    index=False,
    encoding="utf-8-sig"
)


# =====================================================
# 2.3. РЕДКИЕ ПЕРЕХОДЫ — МЕНЕЕ 3%
# =====================================================

rare_transitions = transition_details.loc[
    transition_details["Доля_от_исходной_категории_%"] < 3
].copy()

rare_transitions.to_csv(
    TABLES_DIR / "stage2_rare_transitions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n2.3. РЕДКИЕ ПЕРЕХОДЫ (МЕНЕЕ 3%)")
display(rare_transitions)


# =====================================================
# 2.4. ВЫПАДЕНИЕ И ВОЗВРАЩЕНИЕ РЕСПОНДЕНТОВ
# =====================================================

attrition_results = []

ever_seen = set()

for i, year in enumerate(waves):

    current_ids = set(
        transition_data.loc[
            transition_data["year"] == year, "idind"
        ].dropna()
    )

    if i == 0:
        new_ids = len(current_ids)
        returning_ids = 0
        lost_ids = 0
        retention = np.nan
        attrition_rate = np.nan

    else:
        previous_year = waves[i - 1]

        previous_ids = set(
            transition_data.loc[
                transition_data["year"] == previous_year, "idind"
            ].dropna()
        )

        retained_ids = current_ids & previous_ids
        lost_set = previous_ids - current_ids
        returning_set = (current_ids - previous_ids) & ever_seen

        new_ids = len(current_ids - ever_seen)
        returning_ids = len(returning_set)
        lost_ids = len(lost_set)

        retention = (
            len(retained_ids) / len(previous_ids) * 100
            if len(previous_ids) > 0 else np.nan
        )

        attrition_rate = (
            lost_ids / len(previous_ids) * 100
            if len(previous_ids) > 0 else np.nan
        )

    attrition_results.append({
        "Год": year,
        "Респондентов": len(current_ids),
        "Новых": new_ids,
        "Вернувшихся": returning_ids,
        "Выпавших_с_предыдущей_волны": lost_ids,
        "Сохранность_панели_%": (
            round(retention, 2)
            if not pd.isna(retention) else np.nan
        ),
        "Выпадение_%": (
            round(attrition_rate, 2)
            if not pd.isna(attrition_rate) else np.nan
        )
    })

    ever_seen.update(current_ids)


attrition_table = pd.DataFrame(attrition_results)

attrition_table.to_csv(
    TABLES_DIR / "stage2_panel_attrition.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n2.4. ВЫПАДЕНИЕ И ВОЗВРАЩЕНИЕ РЕСПОНДЕНТОВ")
display(attrition_table)


print("\nВСЕ РАСЧЁТЫ ЭТАПА 2 ЗАВЕРШЕНЫ.")
print("Таблицы сохранены в:", TABLES_DIR)

Начинаем повторную обработку данных для анализа панели...
Обработано строк: 20,000 / 474,600
Обработано строк: 40,000 / 474,600
Обработано строк: 60,000 / 474,600
Обработано строк: 80,000 / 474,600
Обработано строк: 100,000 / 474,600
Обработано строк: 120,000 / 474,600
Обработано строк: 140,000 / 474,600
Обработано строк: 160,000 / 474,600
Обработано строк: 180,000 / 474,600
Обработано строк: 200,000 / 474,600
Обработано строк: 220,000 / 474,600
Обработано строк: 240,000 / 474,600
Обработано строк: 260,000 / 474,600
Обработано строк: 280,000 / 474,600
Обработано строк: 300,000 / 474,600
Обработано строк: 320,000 / 474,600
Обработано строк: 340,000 / 474,600
Обработано строк: 360,000 / 474,600
Обработано строк: 380,000 / 474,600
Обработано строк: 400,000 / 474,600
Обработано строк: 420,000 / 474,600
Обработано строк: 440,000 / 474,600
Обработано строк: 460,000 / 474,600
Обработано строк: 474,600 / 474,600

Обработка завершена.
Количество наблюдений: 393382
Количество уникальных респонде

,year,E,U,I,NA,E_доля_%,U_доля_%,I_доля_%,NA_доля_%
0,1994,4894,0,3411,588,55.03,0.00,38.36,6.61
1,1995,4571,0,3234,612,54.31,0.00,38.42,7.27
2,1996,4382,0,3287,672,52.54,0.00,39.41,8.06
3,1998,4247,635,3713,104,48.82,7.30,42.68,1.20
4,2000,4508,539,3900,127,49.68,5.94,42.98,1.40
5,2001,5089,553,4305,151,50.40,5.48,42.63,1.50
6,2002,5324,542,4489,144,50.71,5.16,42.76,1.37
7,2003,5506,513,4479,138,51.77,4.82,42.11,1.30
8,2004,5564,544,4353,201,52.19,5.10,40.83,1.89
9,2005,5426,452,4260,199,52.49,4.37,41.21,1.93



Переходы 1998 → 2000
Число респондентов с известными состояниями E/U/I: 6862

Количество переходов:


to,E,U,I
from,,,
E,3037,153,310
U,206,102,155
I,327,134,2438



Доли переходов, %:


to,E,U,I
from,,,
E,86.77,4.37,8.86
U,44.49,22.03,33.48
I,11.28,4.62,84.10


... (101 outputs left out)

,Период,Из,В,Количество,Доля_от_исходной_категории_%
46,2004_2005,E,U,141,2.96
55,2005_2006,E,U,101,2.20
64,2006_2007,E,U,110,1.93
73,2007_2008,E,U,114,2.01
79,2007_2008,I,U,114,2.87
82,2008_2009,E,U,169,2.88
91,2009_2010,E,U,170,2.89
100,2010_2011,E,U,206,2.52
106,2010_2011,I,U,155,2.88
109,2011_2012,E,U,177,2.11



2.4. ВЫПАДЕНИЕ И ВОЗВРАЩЕНИЕ РЕСПОНДЕНТОВ


,Год,Респондентов,Новых,Вернувшихся,Выпавших_с_предыдущей_волны,Сохранность_панели_%,Выпадение_%
0,1998,8699,8699,0,0,NaN,NaN
1,2000,9074,2054,0,1679,80.70,19.30
2,2001,10098,1983,283,1242,86.31,13.69
3,2002,10499,1390,367,1356,86.57,13.43
4,2003,10636,1242,324,1429,86.39,13.61
5,2004,10662,1218,429,1621,84.76,15.24
6,2005,10337,789,446,1560,85.37,14.63
7,2006,12490,3155,532,1534,85.16,14.84
8,2007,12290,1281,484,1965,84.27,15.73
9,2008,11862,1042,521,1991,83.80,16.20



ВСЕ РАСЧЁТЫ ЭТАПА 2 ЗАВЕРШЕНЫ.
Таблицы сохранены в: D:\RLMS_VKR\results\tables


In [67]:
print(attrition_table.tail(5).to_string(index=False))

print("\nПроверка последней строки:")
print(attrition_table.iloc[-1].to_dict())

print("\nПроверка численности последних двух волн:")
for year in waves[-2:]:
    ids = transition_data.loc[
        transition_data["year"] == year, "idind"
    ].nunique()
    print(year, ids)

 Год  Респондентов  Новых  Вернувшихся  Выпавших_с_предыдущей_волны  Сохранность_панели_%  Выпадение_%
2021         14581    821          445                         1284                 91.20         8.80
2022         14374   1317          449                         1973                 86.47        13.53
2023         14221    804          375                         1332                 90.73         9.27
2024         14172   1144          344                         1537                 89.19        10.81
2025         13677    723          272                         1490                 89.49        10.51

Проверка последней строки:
{'Год': 2025.0, 'Респондентов': 13677.0, 'Новых': 723.0, 'Вернувшихся': 272.0, 'Выпавших_с_предыдущей_волны': 1490.0, 'Сохранность_панели_%': 89.49, 'Выпадение_%': 10.51}

Проверка численности последних двух волн:
2024 14172
2025 13677
